In [2]:
import pandas as pd
import numpy as np
import os
import calendar
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import optuna
import torch
from optuna.trial import TrialState
from torch.utils.data import Dataset, DataLoader, TensorDataset,Subset
from captum.attr import IntegratedGradients
from captum.attr import visualization as viz
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
from sklearn.linear_model import LinearRegression
import seaborn as sns
plot_template = dict(
    layout=go.Layout({
        "font_size": 18,
        "xaxis_title_font_size": 24,
        "yaxis_title_font_size": 24})
)
import sys
sys.path.append('../..') # add parent path to sys.path
from OneShotSamplesGenerator import gen_one_out_samples
from Dataset import CustomDatasets
from TimeSeriesDataset import TimeSeriesDataset
from CNNRegressor import CNNRegressor,save_model,load_model
from CNNRegressor import DEVICE
from CNNRegressor import train_model
from CNNRegressor import test_model
from CNNRegressor import predict
from CNNRegressor import plot_train_val_loss_cv
from CNNRegressor import save_best_trial
from CNNRegressor import load_best_trial
from CNNRegressor import Objective_CV,Objective
from metrics import PPMAE,LPMAE
import HydroErr as he

class FliterMonthDataset(Dataset):
    def __init__(self, dataset:torch.utils.data.Dataset, index:pd.DatetimeIndex, months:list):
        self.dataset = dataset
        idx_list = []
        for idx,i_ in zip(index,range(self.dataset.y.shape[0])):
            if idx.month in months:
                idx_list.append(i_)
        
        self.subdataset = Subset(self.dataset, idx_list)
  
    def __len__(self):
        return len(self.subdataset)
    
    def __getitem__(self, i):
        X,y = self.subdataset[i]
        return X,y
hydro_stations = [
    'Guide',
    'Xunhua'
]
start_date = '1960-01-01'
end_date = '2019-12-31'

Build CER(CNN) model and estimate natural flow during 1986-2019

In [3]:
for hydro_station in hydro_stations:
    print("-"*20,hydro_station,"-"*20)
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]

    samples,target,features = gen_one_out_samples(df,target_column='flow',mode='simulate',lags_dict=None,lag=12,lead=1)
    # print(samples)


    cal = samples.loc[:'1982-12-31',:]
    test = samples.loc['1983-01-01':'1985-12-31',:]
    pre = samples.loc['1986-01-01':,:]


    X_scaler = MinMaxScaler(feature_range=(0,1))
    Y_scaler = MinMaxScaler(feature_range=(0,1))
    X_scaler.fit(cal[features])
    Y_scaler.fit(cal[[target]])

    cal_X = X_scaler.transform(cal[features])
    test_X = X_scaler.transform(test[features])
    pre_X = X_scaler.transform(pre[features])

    cal_y = Y_scaler.transform(cal[[target]])
    test_y = Y_scaler.transform(test[[target]])
    pre_y = Y_scaler.transform(pre[[target]])

    cal = pd.concat([pd.DataFrame(cal_X,columns=features,index=cal.index),pd.DataFrame(cal_y,columns=[target],index=cal.index)],axis=1)
    test = pd.concat([pd.DataFrame(test_X,columns=features,index=test.index),pd.DataFrame(test_y,columns=[target],index=test.index)],axis=1)
    pre = pd.concat([pd.DataFrame(pre_X,columns=features,index=pre.index),pd.DataFrame(pre_y,columns=[target],index=pre.index)],axis=1)

    cal_dataset = CustomDatasets(cal,target)
    test_dataset = CustomDatasets(test,target)
    pre_dataset = CustomDatasets(pre,target)

    # print(cal_dataset.X.shape, cal_dataset.y.shape)
    # print(test_dataset.X.shape, test_dataset.y.shape)
    # print(pre_dataset.X.shape, pre_dataset.y.shape)

    spring_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[3,4,5])
    summer_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[6, 7, 8])
    autumn_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[9, 10, 11])
    winter_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[12, 1, 2])

    model_path = f'../models/CER/{hydro_station.lower()}/CNN_lag12/'
    if not os.path.exists(model_path):
        os.makedirs(model_path)
    if not os.path.exists(model_path+'best_trial.pickle'):
        objective = Objective(
                train_dataset=cal_dataset,
                val_dataset=test_dataset,
                num_epoch=1000,
                batch_size=32,
                shuffle=True,
                model_path=model_path,
        )
        study = optuna.create_study(
                study_name='example-study',
                direction='minimize',
            )
        study.optimize(objective, n_trials=100)

        pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
        complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])


        print("Study statistics: ")
        print("  Number of finished trials: ", len(study.trials))
        print("  Number of pruned trials: ", len(pruned_trials))
        print("  Number of complete trials: ", len(complete_trials))

        print("Best trial:")
        trial = study.best_trial

        save_best_trial(trial, model_path=model_path)

        save_model(trial, model_path=model_path)

        best_model_state = trial.user_attrs["best_model_state"]


        print("  Value: ", trial.value)

        print("  Params: ")
        for key, value in trial.params.items():
            print("    {}: {}".format(key, value))

    ###########################
    best_trial = load_best_trial(model_file=model_path+'best_trial.pickle')
    model1 = load_model(model_file=model_path+'model.pickle').to(DEVICE)
    cal_loader = DataLoader(cal_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    pre_loader = DataLoader(pre_dataset, batch_size=64, shuffle=False)
    ystar_col = "CER-CNN"
    # print(cal)
    cal_ = cal.loc[cal.index,[target]].copy()
    test_ = test.loc[test.index,[target]].copy()
    pre_ = pre.loc[pre.index,[target]].copy()

    # get shape of cal_loader
    # for X, y in cal_loader:
    #     print(X.shape, y.shape)

    # print(cal_.shape, test_.shape, pre_.shape)

    cal_[ystar_col] = predict(cal_loader, model1).cpu().numpy().squeeze()
    test_[ystar_col] = predict(test_loader, model1).cpu().numpy().squeeze()
    pre_[ystar_col] = predict(pre_loader, model1).cpu().numpy().squeeze()

    # Denormalize the predictions
    cal_[ystar_col] = Y_scaler.inverse_transform(cal_[[ystar_col]])
    test_[ystar_col] = Y_scaler.inverse_transform(test_[[ystar_col]])
    pre_[ystar_col] = Y_scaler.inverse_transform(pre_[[ystar_col]])

    # Denormalize the target values
    cal_[target] = Y_scaler.inverse_transform(cal_[[target]])
    test_[target] = Y_scaler.inverse_transform(test_[[target]])
    pre_[target] = Y_scaler.inverse_transform(pre_[[target]])

    # print(cal_)

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'
    metrics['ME'] = [he.me(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.me(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.male(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.msle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mde(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ed(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ned(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmsle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_range(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_mean(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_iqr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.irmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mase(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.r_squared(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.pearson_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.spearman_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.acc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mapd(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.maape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape2(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D'] = [he.d(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dmod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.drel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.watt_m(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mb_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_mod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_rel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2009(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2012(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.lm_index(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.d1_p(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ve(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sa(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sid(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sga(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.g_mean_diff(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mean_var(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),PPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),LPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]

    df_out = pd.concat((cal_, test_, pre_))[[target, ystar_col]]
    # for c in df_out.columns:
    #     # df_out[c] = df_out[c] * target_std + target_mean
    #     df_out[c] = Y_scaler.inverse_transform(df_out[[c]])

    df_out.to_csv(f'../results/vif_cnn_{hydro_station.lower()}.csv')
    print(df_out)

    metrics.to_csv(f'../results/vif_cnn_metrics_{hydro_station.lower()}.csv')

    # metrics

-------------------- Guide --------------------
             flow_{t+1}      CER-CNN
date                                
1961-01-31   173.013740   203.477081
1961-02-28   167.989418   200.974106
1961-03-31   241.002091   233.198914
1961-04-30   476.851852   498.693451
1961-05-31   673.909797   727.162415
...                 ...          ...
2019-08-31  1005.940000  1481.706299
2019-09-30  1700.630000  1952.280762
2019-10-31  1071.610000  1614.569458
2019-11-30   566.370000   703.196899
2019-12-31   548.870000   343.364288

[708 rows x 2 columns]
-------------------- Xunhua --------------------
             flow_{t+1}      CER-CNN
date                                
1961-01-31   219.011350   214.592300
1961-02-28   214.988426   218.858307
1961-03-31   279.009857   258.787994
1961-04-30   525.077160   533.452454
1961-05-31   701.911589   687.331360
...                 ...          ...
2019-08-31  1214.903226  1574.146118
2019-09-30  1999.066667  1915.202637
2019-10-31  1327.225806  160

Build Extension(CNN) model and estimate natural flow during 1986-2019

In [4]:
for hydro_station in hydro_stations:
    print("-"*20,hydro_station,"-"*20)
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]

    # drop tnh_flow
    df = df.drop(columns=['tnh_flow'])

    samples,target,features = gen_one_out_samples(df,target_column='flow',mode='simulate',lags_dict=None,lag=12,lead=1)
    print(samples)


    cal = samples.loc[:'1982-12-31',:]
    test = samples.loc['1983-01-01':'1985-12-31',:]
    pre = samples.loc['1986-01-01':,:]


    X_scaler = MinMaxScaler(feature_range=(0,1))
    Y_scaler = MinMaxScaler(feature_range=(0,1))
    X_scaler.fit(cal[features])
    Y_scaler.fit(cal[[target]])

    cal_X = X_scaler.transform(cal[features])
    test_X = X_scaler.transform(test[features])
    pre_X = X_scaler.transform(pre[features])

    cal_y = Y_scaler.transform(cal[[target]])
    test_y = Y_scaler.transform(test[[target]])
    pre_y = Y_scaler.transform(pre[[target]])

    cal = pd.concat([pd.DataFrame(cal_X,columns=features,index=cal.index),pd.DataFrame(cal_y,columns=[target],index=cal.index)],axis=1)
    test = pd.concat([pd.DataFrame(test_X,columns=features,index=test.index),pd.DataFrame(test_y,columns=[target],index=test.index)],axis=1)
    pre = pd.concat([pd.DataFrame(pre_X,columns=features,index=pre.index),pd.DataFrame(pre_y,columns=[target],index=pre.index)],axis=1)

    cal_dataset = CustomDatasets(cal,target)
    test_dataset = CustomDatasets(test,target)
    pre_dataset = CustomDatasets(pre,target)

    print(cal_dataset.X.shape, cal_dataset.y.shape)
    print(test_dataset.X.shape, test_dataset.y.shape)
    print(pre_dataset.X.shape, pre_dataset.y.shape)

    spring_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[3,4,5])
    summer_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[6, 7, 8])
    autumn_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[9, 10, 11])
    winter_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[12, 1, 2])

    model_path = f'../models/Extension/{hydro_station.lower()}/CNN_lag12/'
    if not os.path.exists(model_path):
        os.makedirs(model_path)
    if not os.path.exists(model_path+'best_trial.pickle'):
        objective = Objective(
                train_dataset=cal_dataset,
                val_dataset=test_dataset,
                num_epoch=1000,
                batch_size=32,
                shuffle=True,
                model_path=model_path,
        )
        study = optuna.create_study(
                study_name='example-study',
                direction='minimize',
            )
        study.optimize(objective, n_trials=100)

        pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
        complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])


        print("Study statistics: ")
        print("  Number of finished trials: ", len(study.trials))
        print("  Number of pruned trials: ", len(pruned_trials))
        print("  Number of complete trials: ", len(complete_trials))

        print("Best trial:")
        trial = study.best_trial

        save_best_trial(trial, model_path=model_path)

        save_model(trial, model_path=model_path)

        best_model_state = trial.user_attrs["best_model_state"]


        print("  Value: ", trial.value)

        print("  Params: ")
        for key, value in trial.params.items():
            print("    {}: {}".format(key, value))

    ###########################
    best_trial = load_best_trial(model_file=model_path+'best_trial.pickle')
    model1 = load_model(model_file=model_path+'model.pickle').to(DEVICE)
    cal_loader = DataLoader(cal_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    pre_loader = DataLoader(pre_dataset, batch_size=64, shuffle=False)
    ystar_col = "E-CNN"
    # print(cal)
    cal_ = cal.loc[cal.index,[target]].copy()
    test_ = test.loc[test.index,[target]].copy()
    pre_ = pre.loc[pre.index,[target]].copy()

    # get shape of cal_loader
    # for X, y in cal_loader:
    #     print(X.shape, y.shape)

    # print(cal_.shape, test_.shape, pre_.shape)

    cal_[ystar_col] = predict(cal_loader, model1).cpu().numpy().squeeze()
    test_[ystar_col] = predict(test_loader, model1).cpu().numpy().squeeze()
    pre_[ystar_col] = predict(pre_loader, model1).cpu().numpy().squeeze()

    # Denormalize the predictions
    cal_[ystar_col] = Y_scaler.inverse_transform(cal_[[ystar_col]])
    test_[ystar_col] = Y_scaler.inverse_transform(test_[[ystar_col]])
    pre_[ystar_col] = Y_scaler.inverse_transform(pre_[[ystar_col]])

    # Denormalize the target values
    cal_[target] = Y_scaler.inverse_transform(cal_[[target]])
    test_[target] = Y_scaler.inverse_transform(test_[[target]])
    pre_[target] = Y_scaler.inverse_transform(pre_[[target]])

    print(cal_)

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'
    metrics['ME'] = [he.me(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.me(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.male(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.msle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mde(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ed(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ned(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmsle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_range(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_mean(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_iqr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.irmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mase(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.r_squared(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.pearson_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.spearman_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.acc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mapd(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.maape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape2(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D'] = [he.d(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dmod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.drel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.watt_m(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mb_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_mod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_rel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2009(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2012(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.lm_index(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.d1_p(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ve(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sa(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sid(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sga(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.g_mean_diff(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mean_var(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),PPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),LPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]

    df_out = pd.concat((cal_, test_, pre_))[[target, ystar_col]]
    # for c in df_out.columns:
    #     # df_out[c] = df_out[c] * target_std + target_mean
    #     df_out[c] = Y_scaler.inverse_transform(df_out[[c]])

    # print(df_out)
    df_out.to_csv(f'../results/cnn(Extension)_{hydro_station.lower()}.csv')

    metrics.to_csv(f'../results/cnn(Extension)_metrics_{hydro_station.lower()}.csv')

    metrics

-------------------- Guide --------------------
            v10_{t-11}  v10_{t-10}  v10_{t-9}  v10_{t-8}  v10_{t-7}  \
date                                                                  
1961-01-31    0.937511    0.527554   0.934352   0.094989  -0.060234   
1961-02-28    0.527554    0.934352   0.094989  -0.060234  -0.604778   
1961-03-31    0.934352    0.094989  -0.060234  -0.604778  -0.530209   
1961-04-30    0.094989   -0.060234  -0.604778  -0.530209  -0.307727   
1961-05-31   -0.060234   -0.604778  -0.530209  -0.307727  -0.126751   
...                ...         ...        ...        ...        ...   
2019-08-31   -0.053552   -0.344089   0.273150   0.925699   1.068572   
2019-09-30   -0.344089    0.273150   0.925699   1.068572   1.349320   
2019-10-31    0.273150    0.925699   1.068572   1.349320   1.416298   
2019-11-30    0.925699    1.068572   1.349320   1.416298   0.537032   
2019-12-31    1.068572    1.349320   1.416298   0.537032   0.399774   

            v10_{t-6}  v10_{

Build Routing(CNN) model and estimate natural flow during 1986-2019

In [5]:
for hydro_station in hydro_stations:
    print("-"*20,hydro_station,"-"*20)
    df = pd.read_csv(f'../data/{hydro_station.lower()}_vif_modeling_data_1960-2019.csv',parse_dates=['date'],index_col='date')
    df = df.loc[start_date:end_date]

    # preserve tnh_flow and flow
    df = df[['tnh_flow','flow']]

    samples,target,features = gen_one_out_samples(df,target_column='flow',mode='simulate',lags_dict=None,lag=12,lead=1)
    print(samples)


    cal = samples.loc[:'1982-12-31',:]
    test = samples.loc['1983-01-01':'1985-12-31',:]
    pre = samples.loc['1986-01-01':,:]


    X_scaler = MinMaxScaler(feature_range=(0,1))
    Y_scaler = MinMaxScaler(feature_range=(0,1))
    X_scaler.fit(cal[features])
    Y_scaler.fit(cal[[target]])

    cal_X = X_scaler.transform(cal[features])
    test_X = X_scaler.transform(test[features])
    pre_X = X_scaler.transform(pre[features])

    cal_y = Y_scaler.transform(cal[[target]])
    test_y = Y_scaler.transform(test[[target]])
    pre_y = Y_scaler.transform(pre[[target]])

    cal = pd.concat([pd.DataFrame(cal_X,columns=features,index=cal.index),pd.DataFrame(cal_y,columns=[target],index=cal.index)],axis=1)
    test = pd.concat([pd.DataFrame(test_X,columns=features,index=test.index),pd.DataFrame(test_y,columns=[target],index=test.index)],axis=1)
    pre = pd.concat([pd.DataFrame(pre_X,columns=features,index=pre.index),pd.DataFrame(pre_y,columns=[target],index=pre.index)],axis=1)

    cal_dataset = CustomDatasets(cal,target)
    test_dataset = CustomDatasets(test,target)
    pre_dataset = CustomDatasets(pre,target)

    print(cal_dataset.X.shape, cal_dataset.y.shape)
    print(test_dataset.X.shape, test_dataset.y.shape)
    print(pre_dataset.X.shape, pre_dataset.y.shape)

    spring_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[3,4,5])
    summer_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[6, 7, 8])
    autumn_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[9, 10, 11])
    winter_dataset = FliterMonthDataset(dataset=cal_dataset, index=cal.index, months=[12, 1, 2])

    model_path = f'../models/Routing/{hydro_station.lower()}/CNN_lag12/'
    if not os.path.exists(model_path):
        os.makedirs(model_path)
    if not os.path.exists(model_path+'best_trial.pickle'):
        objective = Objective(
                train_dataset=cal_dataset,
                val_dataset=test_dataset,
                num_epoch=1000,
                batch_size=32,
                shuffle=True,
                model_path=model_path,
        )
        study = optuna.create_study(
                study_name='example-study',
                direction='minimize',
            )
        study.optimize(objective, n_trials=100)

        pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
        complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])


        print("Study statistics: ")
        print("  Number of finished trials: ", len(study.trials))
        print("  Number of pruned trials: ", len(pruned_trials))
        print("  Number of complete trials: ", len(complete_trials))

        print("Best trial:")
        trial = study.best_trial

        save_best_trial(trial, model_path=model_path)

        save_model(trial, model_path=model_path)

        best_model_state = trial.user_attrs["best_model_state"]


        print("  Value: ", trial.value)

        print("  Params: ")
        for key, value in trial.params.items():
            print("    {}: {}".format(key, value))

    ###########################
    best_trial = load_best_trial(model_file=model_path+'best_trial.pickle')
    model1 = load_model(model_file=model_path+'model.pickle').to(DEVICE)
    cal_loader = DataLoader(cal_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    pre_loader = DataLoader(pre_dataset, batch_size=64, shuffle=False)
    ystar_col = "R-CNN"
    # print(cal)
    cal_ = cal.loc[cal.index,[target]].copy()
    test_ = test.loc[test.index,[target]].copy()
    pre_ = pre.loc[pre.index,[target]].copy()

    # get shape of cal_loader
    # for X, y in cal_loader:
    #     print(X.shape, y.shape)

    # print(cal_.shape, test_.shape, pre_.shape)

    cal_[ystar_col] = predict(cal_loader, model1).cpu().numpy().squeeze()
    test_[ystar_col] = predict(test_loader, model1).cpu().numpy().squeeze()
    pre_[ystar_col] = predict(pre_loader, model1).cpu().numpy().squeeze()

    # Denormalize the predictions
    cal_[ystar_col] = Y_scaler.inverse_transform(cal_[[ystar_col]])
    test_[ystar_col] = Y_scaler.inverse_transform(test_[[ystar_col]])
    pre_[ystar_col] = Y_scaler.inverse_transform(pre_[[ystar_col]])

    # Denormalize the target values
    cal_[target] = Y_scaler.inverse_transform(cal_[[target]])
    test_[target] = Y_scaler.inverse_transform(test_[[target]])
    pre_[target] = Y_scaler.inverse_transform(pre_[[target]])

    print(cal_)

    metrics = pd.DataFrame(index=['cal','test'])
    metrics.index.name = 'period'
    metrics['ME'] = [he.me(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.me(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAE'] = [he.mae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSE'] = [he.mse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MLE'] = [he.mle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MALE'] = [he.male(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.male(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MSLE'] = [he.msle(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.msle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDE'] = [he.mde(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mde(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDAE'] = [he.mdae(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdae(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MDSE'] = [he.mdse(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mdse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ED'] = [he.ed(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ed(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NED'] = [he.ned(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ned(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSE'] = [he.rmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['RMSLE'] = [he.rmsle(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.rmsle(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_RANGE'] = [he.nrmse_range(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_range(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_MEAN'] = [he.nrmse_mean(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_mean(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NRMSE_IQR'] = [he.nrmse_iqr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nrmse_iqr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['IRMSE'] = [he.irmse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.irmse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MASE'] = [he.mase(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mase(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['R_SQUARED'] = [he.r_squared(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.r_squared(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PEARSON_R'] = [he.pearson_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.pearson_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SPEARMAN_R'] = [he.spearman_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.spearman_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['ACC'] = [he.acc(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.acc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPE'] = [he.mape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAPD'] = [he.mapd(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mapd(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MAAPE'] = [he.maape(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.maape(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE1'] = [he.smape1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SMAPE2'] = [he.smape2(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.smape2(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D'] = [he.d(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1'] = [he.d1(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.d1(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DMOD'] = [he.dmod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dmod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DREL'] = [he.drel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.drel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['DR'] = [he.dr(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.dr(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['WATT_M'] = [he.watt_m(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.watt_m(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MB_R'] = [he.mb_r(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.mb_r(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE'] = [he.nse(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_MOD'] = [he.nse_mod(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_mod(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['NSE_REL'] = [he.nse_rel(observed_array=cal_[target], simulated_array=cal_[ystar_col]), he.nse_rel(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2009'] = [he.kge_2009(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2009(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['KGE_2012'] = [he.kge_2012(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.kge_2012(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['LM_INDEX'] = [he.lm_index(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.lm_index(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['D1_P'] = [he.d1_p(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.d1_p(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['VE'] = [he.ve(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.ve(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SA'] = [he.sa(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sa(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SC'] = [he.sc(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sc(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SID'] = [he.sid(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sid(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['SGA'] = [he.sga(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.sga(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MHE'] = [he.h1_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_MAHE'] = [he.h1_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H1_RMSHE'] = [he.h1_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h1_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MHE'] = [he.h2_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_MAHE'] = [he.h2_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H2_RMSHE'] = [he.h2_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h2_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MHE'] = [he.h3_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_MAHE'] = [he.h3_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H3_RMSHE'] = [he.h3_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h3_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MHE'] = [he.h4_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_MAHE'] = [he.h4_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H4_RMSHE'] = [he.h4_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h4_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MHE'] = [he.h5_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_MAHE'] = [he.h5_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H5_RMSHE'] = [he.h5_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h5_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MHE'] = [he.h6_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_MAHE'] = [he.h6_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H6_RMSHE'] = [he.h6_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h6_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MHE'] = [he.h7_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_MAHE'] = [he.h7_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H7_RMSHE'] = [he.h7_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h7_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MHE'] = [he.h8_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_MAHE'] = [he.h8_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H8_RMSHE'] = [he.h8_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h8_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MHE'] = [he.h10_mhe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mhe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_MAHE'] = [he.h10_mahe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_mahe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['H10_RMSHE'] = [he.h10_rmshe(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.h10_rmshe(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['G_MEAN_DIFF'] = [he.g_mean_diff(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.g_mean_diff(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['MEAN_VAR'] = [he.mean_var(observed_array=cal_[target], simulated_array=cal_[ystar_col]),he.mean_var(observed_array=test_[target], simulated_array=test_[ystar_col]),]
    metrics['PPMAE'] = [PPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),PPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]
    metrics['LPMAE'] = [LPMAE(y_true=cal_[target],y_pred=cal_[ystar_col]),LPMAE(y_true=test_[target],y_pred=test_[ystar_col]),]

    df_out = pd.concat((cal_, test_, pre_))[[target, ystar_col]]
    # for c in df_out.columns:
    #     # df_out[c] = df_out[c] * target_std + target_mean
    #     df_out[c] = Y_scaler.inverse_transform(df_out[[c]])

    # print(df_out)
    df_out.to_csv(f'../results/cnn(Routing)_{hydro_station.lower()}.csv')

    metrics.to_csv(f'../results/cnn(Routing)_metrics_{hydro_station.lower()}.csv')

    metrics

-------------------- Guide --------------------
            tnh_flow_{t-11}  tnh_flow_{t-10}  tnh_flow_{t-9}  tnh_flow_{t-8}  \
date                                                                           
1961-01-31       114.994026       121.008940      158.900836      209.992284   
1961-02-28       121.008940       158.900836      209.992284      252.016129   
1961-03-31       158.900836       209.992284      252.016129      554.012346   
1961-04-30       209.992284       252.016129      554.012346     1080.122461   
1961-05-31       252.016129       554.012346     1080.122461     1210.050777   
...                     ...              ...             ...             ...   
2019-08-31      1298.387097      2098.333333     1399.193548      692.766667   
2019-09-30      2098.333333      1399.193548      692.766667      345.806452   
2019-10-31      1399.193548       692.766667      345.806452      264.129032   
2019-11-30       692.766667       345.806452      264.129032      270.17